In [3]:
#omport libraries
import pandas as pd
import numpy as np

In [4]:
#load datasets
train = pd.read_csv("train_LZdllcl.csv")
test = pd.read_csv("test_2umaH9m.csv")
sample = pd.read_csv("sample_submission_M0L0uXE.csv")

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54808 entries, 0 to 54807
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           54808 non-null  int64  
 1   department            54808 non-null  object 
 2   region                54808 non-null  object 
 3   education             52399 non-null  object 
 4   gender                54808 non-null  object 
 5   recruitment_channel   54808 non-null  object 
 6   no_of_trainings       54808 non-null  int64  
 7   age                   54808 non-null  int64  
 8   previous_year_rating  50684 non-null  float64
 9   length_of_service     54808 non-null  int64  
 10  KPIs_met >80%         54808 non-null  int64  
 11  awards_won?           54808 non-null  int64  
 12  avg_training_score    54808 non-null  int64  
 13  is_promoted           54808 non-null  int64  
dtypes: float64(1), int64(8), object(5)
memory usage: 5.9+ MB


In [6]:
train.isnull().sum()

employee_id                0
department                 0
region                     0
education               2409
gender                     0
recruitment_channel        0
no_of_trainings            0
age                        0
previous_year_rating    4124
length_of_service          0
KPIs_met >80%              0
awards_won?                0
avg_training_score         0
is_promoted                0
dtype: int64

In [7]:
# Fill missing values
train = train.fillna(0)
test = test.fillna(0)

In [9]:
#convert categorical to number
train["education"] = train["education"].fillna(train["education"].mode()[0])
test["education"] = test["education"].fillna(test["education"].mode()[0])


In [11]:
# Fill numerical column
train["previous_year_rating"] = train["previous_year_rating"].fillna(train["previous_year_rating"].median())
test["previous_year_rating"] = test["previous_year_rating"].fillna(test["previous_year_rating"].median())

In [15]:
# Feature Engineering (important for better performance)
train["total_score"] = train["avg_training_score"] * train["previous_year_rating"]
test["total_score"] = test["avg_training_score"] * test["previous_year_rating"]

train["age_group"] = train["age"] // 10
test["age_group"] = test["age"] // 10

In [18]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['department', 'region', 'education', 'gender', 'recruitment_channel']

for col in cat_cols:
    le = LabelEncoder()
    
    # Convert to string
    train[col] = train[col].astype(str)
    test[col] = test[col].astype(str)
    
    combined = pd.concat([train[col], test[col]])
    le.fit(combined)
    
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

In [19]:
#Split features and target
X = train.drop("is_promoted", axis=1)
y = train["is_promoted"]

In [20]:

#Train-test split
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [21]:
 !pip install xgboost

In [22]:
#Train model (XGBoost)
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=5,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [23]:
predictions = model.predict(test)


In [24]:
sample["is_promoted"] = predictions
sample.to_csv("final_submission.csv", index=False)

In [25]:
sample.head()

,employee_id,is_promoted
0,8724,0
1,74430,0
2,72255,0
3,38562,0
4,64486,0
